In this tutorial/guide, we will be improving and evaluating prompts that we send to the Anthropic API (this content is derived from Anthropic skilljar site)

##Prompt Engineering vs Prompt Evaluation

Prompt Engineering is like crafting effective prompts. Includes techniques like:
- Multishot prompting
- Structuring with XML tags
- Many other best practices
We use these techniques to help Claude understand exactly what we are asking it do and how we want it to respond

Prompt Evaluation is measuring the effectiveness of the written/crafted prompts through some automated testing. Here's how it usually works:
- Test against expected answers
- Compare different versions of the same prompt
- Review outputs for errors

Best practice for drafting a prompt is to run the prompt through an evaluation pipeline, score it and then iterate on the prompt. This way we can:
- Identify weaknesses before they become production issues
- Compare the different prompt versions objectively
- Iterate with confidence based on measurable improvements
- Build more reliable AI applications

A typical prompt evaluation worklow follows 5 steps:
1. Draft a prompt
2. Create an Eval Dataset
3. Feed through Claude
4. Feed through a Grader
5. Change Prompt and Repeat

1. Drafting a prompt

In [ ]:
prompt = f"""
Please answer the user's question:
{question}
"""

2. Create an Eval Dataset

This contains sample inputs that represents the types of questions or requests your prompt will handle in production.



3. Feed through Claude

Take each question from your eval dataset and merge it with your prompt template to create complete prompts. Then send each one to Claude to get responses

4. Feed through a Grader

The grader will evaluate the quality of Claude's responses by examining both the original question and Claude's answer. This step will perform objective scoring on a scale from 1 to 10
10 - perfect answer, 1- poor answer


5. Change Prompt and Repeat

Now that we have a baseline score, we can modify the prompt and run the entire process again (1-4) to see if the changes we made improve the score/performance

#Generating Test Data with Code

Let's create a function that automatically generates our test dataset. The following are helper functions for working with Claude

In [ ]:
def add_user_message(messages, text):
  user_message = {"role": "user", "content": text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {"role": "assistant", "content": text}
  messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
  params = {
      "model": model,
      "max_tokens": 1000,
      "messages": messages,
      "temperature": temperature
  }
  if system:
    params["system"] = system
  if stop_sequences:
    params["stop_sequences"] = stop_sequences

  response = client.messages.create(**params)
  return response.content[0].text

and now the dataset generation function...

In [ ]:
def generate_dataset():
  prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be
    used to evaluate prompts that generate Python, JSON, or Regex specifically
    for AWS-related tasks. Generate an array of JSON objects, each representing
    task that requires Python, JSON, or Regex to complete.

    Example output:
    ```json
    [
      {
        "task": "Description of task",
      },
      ...additional
    ]
    ```
    * Focus on tasks that can be solved by writing a single Python function, a
      single JSON object, or a single regex
    * Focus on tasks that do not require writing much code

    Please generate 3 objects
  """

#Testing the dataset

Running the function should return three different test cases covering our target outputs- Python functions, JSON configurationss, and regular expressions for AWS-specific tasks

In [ ]:
dataset = generate_dataset()
print(dataset)

#Saving the Dataset

Once we have our dataset, we'll save it to a file so we can easily load it later during evaluation. The following code will create a dataset.json file in the same directory as your notebook containing your list of tasks ready for prompt evaluation.